# A07 · Tokens, Embeddings & Positions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a07_tokens_embeddings.ipynb)

A language model never sees text, only integers. Learn how byte-pair encoding builds a vocabulary, why an embedding lookup is a matrix multiply, and how models know word order through sinusoidal and rotary position encodings.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a07). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Run the BPE merge algorithm by hand on a small corpus.
- Show that an embedding lookup equals one-hot $\times E$, and count its parameters.
- Compute sinusoidal position encodings and explain RoPE as a rotation.
- Measure tokens-per-word for English vs Malay.

## The math

### BPE merges on a toy corpus

$$\text{merge}^* = \arg\max_{(a,b)} \text{count}(a\,b)$$

Corpus: low ×5, lower ×2, newest ×6, widest ×3 (with an end-of-word marker </w>).

**Worked example**

1. merge 1: $(e, s)$, count 9 (from newest (6) + widest (3))
2. merges 2–3: $(es, t)$, $(est, \text{</w>})$ (the suffix "est" becomes one token)
3. merges 4–5: $(l, o)$, $(lo, w)$ with count 7 ("low" becomes one token)
4. merges 6–8: $(n,e)$, $(ne,w)$, $(new, est\text{</w>})$ ("newest" is now a single token; "widest" = w·i·d·est)

### Embedding = one-hot × matrix

$$x_i = \mathbf{e}_i^{\top} E \in \mathbb{R}^{d}, \qquad |E| = V\cdot d$$

GPT-2 small: V = 50,257, d = 768.

**Worked example**

1. $|E| = 50{,}257 \times 768 = 38{,}597{,}376$ (31% of GPT-2 small's 124M parameters)
2. gradient: only the rows of tokens in the batch get non-zero updates (sparse rows, dense optimizer state)

### Sinusoidal position encoding

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \qquad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

Example with $d = 4$, position 1.

**Worked example**

1. $i = 0$: frequency $1$ → $(\sin 1, \cos 1) = (0.8415,\ 0.5403)$
2. $i = 1$: $10000^{2/4} = 100$ → $(\sin 0.01, \cos 0.01) = (0.0100,\ 1.0000)$ (slow dimensions change little per position)
3. $PE_1 = (0.8415,\ 0.5403,\ 0.0100,\ 1.0000)$

### RoPE: relative position by rotation

$$R_m = \begin{pmatrix}\cos m\theta & -\sin m\theta\\ \sin m\theta & \cos m\theta\end{pmatrix},\quad (R_m q)^{\top}(R_n k) = q^{\top}R_{n-m}\,k$$

Each 2-D pair of dimensions $j$ uses frequency $\theta_j = 10000^{-2j/d}$. Since rotations compose, the attention score depends on the offset $n - m$ only.

**Worked example**

1. $q = k = (1, 0),\ \theta = 0.5$: positions 3 and 5 → angle difference $2\cdot0.5 = 1$ rad
2. score $= \cos 1 = 0.540$ (same for positions 10 and 12)

In [ ]:
%pip install -q tiktoken

## Lab: Build BPE, then inspect a real tokenizer

Run BPE by hand, verify the embedding identity, compute positional encodings, and count Malay vs English tokens.

### Step 1

BPE from scratch: the output matches the worked example.

In [ ]:
from collections import Counter
corpus = "low " * 5 + "lower " * 2 + "newest " * 6 + "widest " * 3
words = Counter(corpus.split())
vocab = {tuple(w) + ('</w>',): c for w, c in words.items()}

def pair_counts(vocab):
    pairs = Counter()
    for sym, c in vocab.items():
        for a, b in zip(sym, sym[1:]):
            pairs[(a, b)] += c
    return pairs

def merge(vocab, pair):
    out = {}
    for sym, c in vocab.items():
        s, i = [], 0
        while i < len(sym):
            if i < len(sym) - 1 and (sym[i], sym[i + 1]) == pair:
                s.append(sym[i] + sym[i + 1]); i += 2
            else:
                s.append(sym[i]); i += 1
        out[tuple(s)] = c
    return out

for step in range(8):
    pairs = pair_counts(vocab)
    best = max(pairs, key=pairs.get)
    vocab = merge(vocab, best)
    print(step + 1, best, pairs[best])
print(vocab)

### Step 2

An embedding lookup is a one-hot matmul.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
V, d = 10, 4
emb = nn.Embedding(V, d)
ids = torch.tensor([3, 7])
print(torch.allclose(emb(ids), F.one_hot(ids, V).float() @ emb.weight))   # True

### Step 3

Sinusoidal encodings: position 1 matches the worked example.

In [ ]:
def sinusoidal(n_pos, d):
    pos = torch.arange(n_pos).unsqueeze(1)          # (n_pos, 1)
    i = torch.arange(0, d, 2)                        # even dims: 0, 2, ...
    angle = pos / (10000 ** (i / d))
    pe = torch.zeros(n_pos, d)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe
print(sinusoidal(2, 4)[1])     # tensor([0.8415, 0.5403, 0.0100, 1.0000])

### Step 4

How many GPT-2 tokens do English and Malay need? (`pip install tiktoken`)

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
for text in ["Good morning, how are you?", "Selamat pagi, apa khabar?",
             "cloud computing architecture", "seni bina pengkomputeran awan"]:
    ids = enc.encode(text)
    print(f"{len(ids):2d} tokens  {[enc.decode([i]) for i in ids]}")

### Clean up

Nothing to clean up.

## Check yourself

**1. Vocabulary 32,000, $d = 4096$, untied input and output matrices. Parameters in both?**

A. 131M  
B. 262M  
C. 32M  
D. 4.1B

<details><summary>Answer</summary>

**B.** $2 \cdot 32{,}000 \cdot 4096 = 262{,}144{,}000$.

</details>

**2. What does BPE merge at each step?**

A. The longest word  
B. The most frequent adjacent symbol pair  
C. Random characters  
D. Words with the same meaning

<details><summary>Answer</summary>

**B.** Greedy: count all adjacent pairs weighted by word frequency, merge the top one, repeat.

</details>

**3. With RoPE, the attention score between positions $m$ and $n$ depends on…**

A. $m + n$  
B. only $n - m$ (plus content)  
C. $m\cdot n$  
D. Nothing positional

<details><summary>Answer</summary>

**B.** $R_m^\top R_n = R_{n-m}$: relative position is built into the dot product.

</details>